# 🔨 Build a Feature, Stage by Stage

**The exercise:** take ONE demo feature — `TOY-2`: *add `toybox/stats.py` with `mean()` and
`median()` plus tests* — and walk it through **every stage of the development loop by hand.**
You are the loop. The daemon that orchestrates this later is these exact function calls in a
`while` loop, so after this walkthrough it holds no mystery.

Follow the map in `ranch_lab.ipynb`'s opening diagram:
**triage → scope → propose → decide → execute → verify → ship** — blue stages are plain
Python you run offline for free; green stages spawn a real Claude session (flagged 🔴 LIVE).

Run cells top to bottom. Sandbox-isolated; only the throwaway `toybox` repo is touched.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from datetime import datetime, timedelta, timezone

os.environ["RANCH_HOME"] = str(Path.home() / ".ranch-sandbox")
RANCH  = Path.home() / "code/citemed/ranch"
TOYBOX = Path.home() / "code/citemed/toybox"
sys.path.insert(0, str(RANCH))

from ranch.db import init_db, db_session
init_db()

def ranch_cli(*args):
    r = subprocess.run([str(RANCH / ".venv/bin/ranch"), *args],
                       capture_output=True, text=True, env={**os.environ})
    print((r.stdout or "") + (r.stderr or ""))

TICKET = "TOY-2"
print("sandbox ready — building", TICKET)

## Stage 1 — TRIAGE (plain Python, free) 🔵

*"Which ticket should the worker pick up?"* — answered by a **scoring formula**, not a model:
status (+30 in-progress / +20 to-do / 0 blocked) · has-Figma-link (+20) ·
has-acceptance-criteria (+15) · priority ladder · age (log-curve, max +10).

Below: a synthetic 3-ticket board. Predict the winner, then run it.

In [ ]:
from ranch.triage import JiraTicket, score_ticket

def mk(key, summary, status, cat, prio, days_old, desc, figma=False):
    now = datetime.now(timezone.utc)
    return JiraTicket(key=key, summary=summary, status=status, status_category=cat,
                      priority=prio, created=now - timedelta(days=days_old), updated=now,
                      description=desc, has_figma_link=figma)

board = [
    mk("TOY-2", "Add stats module", "To Do", "new", "High", 3,
       "Acceptance criteria:\n1. mean(xs) should return the average\n2. median(xs) must handle even-length lists"),
    mk("TOY-3", "Vague idea, no spec", "To Do", "new", None, 1, "make it better somehow"),
    mk("TOY-4", "Blocked redesign", "Blocked", "indeterminate", "Highest", 40,
       "Redesign everything", figma=True),
]

for t in board:
    s = score_ticket(t, in_flight_ticket_keys=set())
    print(f"{t.key}  total={s.total:5.1f}   (status={s.status:+.0f}  design={s.design_present:+.0f}  "
          f"AC={s.ac_clarity:+.0f}  priority={s.priority:+.0f}  age={s.age:+.1f})")

In [ ]:
# 🧪 PLAYGROUND — change the inputs, re-run, watch the ranking move:
#  · give TOY-3 acceptance criteria → +15
#  · unblock TOY-4 (status="In Progress", cat="indeterminate") → +30
#  · mark TOY-2 in-flight → score collapses to -1000 (never double-pick):
print(score_ticket(board[0], in_flight_ticket_keys={"TOY-2"}).total)

## Stage 2 — SCOPE (plain Python, free) 🔵

*"What context does the worker need before starting?"* Normally
`ranch scope TOY-2 --save` fetches the epic, sister tickets, and open PRs from
Jira/Bitbucket and writes a markdown bundle to `$RANCH_HOME/scopes/TOY-2.md`.

Toybox has no Jira — so **you write the bundle by hand**, which teaches the more
important fact: *the bundle is just a markdown file*. Propose consumes whatever is there.

In [ ]:
from ranch.scope import SCOPES_DIR

SCOPES_DIR.mkdir(parents=True, exist_ok=True)
scope_md = f"""# Scope — {TICKET}: Add stats module

## Ticket
Add `toybox/stats.py` with `mean(xs)` and `median(xs)`.

## Acceptance criteria
1. `mean(xs)` returns the arithmetic average; raises `ValueError` on an empty list.
2. `median(xs)` handles odd AND even lengths; raises `ValueError` on an empty list.
3. Tests live in `tests/test_stats.py`; `python3 -m pytest -q` fully green.

## Repo notes
- Pure stdlib. Follow the existing style of `toybox/mathx.py`.
- No PR platform: push the branch, then stop.
"""
(SCOPES_DIR / f"{TICKET}.md").write_text(scope_md)
print("saved →", SCOPES_DIR / f"{TICKET}.md")

In [ ]:
# What the propose session will actually receive (brief = scope + instructions):
from ranch.propose import build_propose_brief
print(build_propose_brief(TICKET, scope_md)[:900], "…")

## Stage 3 — PROPOSE (🔴 LIVE — SDK session #1)

A **short, read-only** Claude session (file-modification tools disabled, 180 s budget):
it reads the code + your scope bundle and produces a **plan + machine-runnable acceptance
checks**, then parks. No code gets written — that's the safety property of this stage.

Cost: one small session. Flip `LIVE = True`.

In [ ]:
LIVE = False   # ← flip to True to run the real sessions in this notebook

if LIVE:
    ranch_cli("propose", TICKET, "--agent", "toy", "--auto-approve")
else:
    print("LIVE=False — flip it when you're ready.")

In [ ]:
# The artifact: a PARKED dossier holding the plan + acceptance contract.
if LIVE:
    from ranch.models import Run, Dossier
    with db_session() as db:
        prun = (db.query(Run).filter_by(agent="toy", ticket=TICKET)
                  .order_by(Run.id.desc()).first())
        d = (db.query(Dossier).filter_by(run_id=prun.id)
               .order_by(Dossier.id.desc()).first())
        propose_run_id, payload = prun.id, json.loads(d.payload_json)
    print("propose run:", propose_run_id, "| dossier state:", payload["state"])
    print("\nPLAN:");       [print("  -", s["step"]) for s in payload.get("plan", [])]
    print("\nACCEPTANCE:"); [print("  -", c) for c in payload.get("acceptance", [])]
else:
    print("LIVE=False")

## Stage 4 — DECIDE (you, the gate) 🟠

The plan is parked. Nothing proceeds until you say so. Approving writes an **Interjection**;
then you'll do *exactly what the daemon does*: find the approved plan and consume it
atomically (so the same approval can never fire twice).

Prefer to push back instead? `ranch_cli("reject", str(propose_run_id), "--reason", "…")`
and re-run Stage 3 — the feedback is woven into the revised brief.

In [ ]:
if LIVE:
    ranch_cli("approve", str(propose_run_id))
    from ranch.hand import _find_approved_parked_propose
    approved = _find_approved_parked_propose("toy")     # ← the daemon's own finder
    print("found + consumed approval for:", approved.propose_run.ticket)
    print("second call returns:", _find_approved_parked_propose("toy"))
else:
    print("LIVE=False")

## Stage 5 — EXECUTE (🔴 LIVE — SDK session #2)

Now the working session: it inherits the approved plan + acceptance contract (no
re-deriving), then runs **plan → TDD → QA → pre-push** *inside one session*.
`plan_ready`/`tests_green` auto-approve (the plan was already vetted — same setting the
daemon uses); **`pre_push` is a real stop**.

The cell below **streams the whole session live into the notebook**. While it's running,
the notebook kernel is busy — so when it parks at `pre_push`, approve **from a terminal**:
```bash
RANCH_HOME=~/.ranch-sandbox ~/code/citemed/ranch/.venv/bin/ranch approve <exec_run_id>
```
(the running session polls for your decision every 500 ms and then pushes).

In [ ]:
if LIVE:
    from ranch.hand import _build_execute_brief, _create_execute_run
    brief = _build_execute_brief(TICKET, approved.parked_payload)
    exec_run_id = _create_execute_run(agent="toy", cwd=TOYBOX, ticket=TICKET,
                                      brief=brief, parked_payload=approved.parked_payload)
    print("execute run id:", exec_run_id, " ← use this in the terminal approve command")
else:
    print("LIVE=False")

In [ ]:
if LIVE:
    from ranch.runner.orchestrator import Orchestrator
    orch = Orchestrator(agent="toy", cwd=TOYBOX, ticket=TICKET, brief=brief,
                        auto_approve_kinds={"plan_ready", "tests_green"},
                        budget_seconds=900)
    orch.run_id = exec_run_id
    await orch.run()          # streams live; approve pre_push from your terminal
else:
    print("LIVE=False")

## Stage 6 — VERIFY (plain Python, free) 🔵

The acceptance checks from Stage 3 are **not opinions — they execute**. During Stage 5 the
agent already ran them via `run_acceptance` (the self-judge); here you run the *same judge*
yourself, outside any session, against the worktree. This is also the seam where the future
independent reviewer (the "Brand Inspector") plugs in — same checks, different judge.

In [ ]:
from ranch.judge import run_acceptance
from ranch.runner.messages import AcceptanceCheck

raw = (approved.parked_payload.get("acceptance") if LIVE else None) or [
    {"kind": "script", "name": "pytest fully green", "cmd": "python3 -m pytest -q",
     "pass_pattern": "passed"},
]
checks = [AcceptanceCheck(**c) for c in raw]
result = run_acceptance(checks, TOYBOX)
for r in result.results:
    print(r)

## Stage 7 — SHIP (plain Python, free) 🔵

`ranch pr draft` renders a PR title + body **from the paperwork the run left behind**
(dossiers, checkpoints, acceptance results) — no model call, no remote call.
`ranch pr open` would then fire `bb`/`gh` for real (toybox has no platform, so draft only).

In [ ]:
if LIVE:
    ranch_cli("pr", "draft", str(exec_run_id))
    print(subprocess.run(["git", "-C", str(Path.home()/"code/citemed/toybox-origin.git"),
                          "branch", "-a"], capture_output=True, text=True).stdout)
else:
    print("LIVE=False — after a live run you'd see the rendered PR + the pushed branch here.")

## Coda — you just were the orchestrator

Every action you took by hand maps 1:1 to a line in the daemon's poll loop:

| You did | The daemon's version |
|---|---|
| Ran triage + picked TOY-2 | `_discover_and_queue()` → operator kickoff |
| Wrote the scope bundle | `scope_fn(ticket)` (`ranch scope --save`) |
| Ran propose | `propose_fn(ticket)` |
| Approved + consumed the plan | `_find_approved_parked_propose()` |
| Built brief + execute run | `_build_execute_brief` → `_create_execute_run` |
| Ran the orchestrator + approved `pre_push` | `execute_fn(approved)` + your `ranch approve` |
| Ran the judge | the in-session `run_acceptance` call |
| Drafted the PR | `ranch pr draft` |

**Orchestration = this notebook in a `while` loop.** When doing it by hand feels boring
rather than confusing, you're ready for `ranch hand start toy` — and for scaling it
horizontally (one loop per hand: jeffy, arnold, max, kesha). Not before.